# Multiple Linear Regression in PySpark

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
import matplotlib.pyplot as plt


## 1. Create Spark session

In [ ]:
spark = SparkSession.builder     .appName("MultipleLinearRegression")     .getOrCreate()


## 2. Generate synthetic data for multiple regression

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 100
X1 = np.random.rand(n)
X2 = np.random.rand(n)
y = 4 + 3*X1 + 5*X2 + np.random.randn(n)

df_pd = pd.DataFrame({'X1': X1, 'X2': X2, 'y': y})
df_spark = spark.createDataFrame(df_pd)
df_spark.show(5)


## 3. Assemble features into a single vector column

In [ ]:
assembler = VectorAssembler(inputCols=["X1", "X2"], outputCol="features")
assembled = assembler.transform(df_spark)


## 4. Fit the linear regression model

In [ ]:
lr = LinearRegression(featuresCol="features", labelCol="y")
model = lr.fit(assembled)

# Coefficients and intercept
print("Coefficients:", model.coefficients)
print("Intercept:", model.intercept)


## 5. Evaluate model performance

In [ ]:
training_summary = model.summary
print("RMSE:", training_summary.rootMeanSquaredError)
print("R^2:", training_summary.r2)


## 6. Plot predicted vs actual values

In [ ]:
predictions = model.transform(assembled).select("y", "prediction")
pdf = predictions.toPandas()

plt.figure(figsize=(6, 6))
plt.scatter(pdf["y"], pdf["prediction"], alpha=0.6)
plt.plot([pdf["y"].min(), pdf["y"].max()], [pdf["y"].min(), pdf["y"].max()], color='red')
plt.xlabel("Actual y")
plt.ylabel("Predicted y")
plt.title("Actual vs Predicted")
plt.grid(True)
plt.show()
